# Diagnostic: Why are predictions zero?
1. Check feature dimensions
2. Check probability distribution
3. Check label loading
4. Check video matching

In [ ]:
# Setup
from google.colab import drive
drive.mount('/content/drive')

import os, numpy as np, pandas as pd
import xgboost as xgb
from sklearn.preprocessing import StandardScaler

BASE = '/content/drive/MyDrive/standup4ai'
FEAT_DIR = BASE + '/features_221'
LABEL_DIR = BASE + '/seq-Standup4AI/dataset/en_uk/emnlp+jahak/train'

print('Setup done')
print('BASE:', BASE)
print('FEAT_DIR:', FEAT_DIR)
print('LABEL_DIR:', LABEL_DIR)


In [ ]:
# 1. Check feature dimensions
feat_files = sorted([f for f in os.listdir(FEAT_DIR) if f.endswith('_features.npy')])
print('Feature files:', len(feat_files))

# Load first file and check shape
sample = np.load(FEAT_DIR + '/' + feat_files[0])
print('Sample shape:', sample.shape)
print('Expected: (n_chunks, 791)')

# Check all files have same dim
dims = []
for f in feat_files[:10]:
    d = np.load(FEAT_DIR + '/' + f)
    dims.append(d.shape[1])
print('Feature dims (first 10):', set(dims))


In [ ]:
# 2. Check which videos have labels
feat_vids = set(f.replace('_features.npy', '') for f in feat_files)
print('Feature videos:', len(feat_vids))

label_files = [f for f in os.listdir(LABEL_DIR) if f.endswith('.csv')]
label_vids = set(f.replace('.csv', '') for f in label_files)
print('Label files:', len(label_files))

overlap = feat_vids & label_vids
print('Overlap:', len(overlap))

missing_labels = feat_vids - label_vids
print('Feature videos missing labels:', len(missing_labels))
if missing_labels:
    print('  Examples:', list(missing_labels)[:3])


In [ ]:
# 3. Load data and check training
def parse_timestamp(ts_str):
    ts_str = str(ts_str).strip()
    try:
        parts = ts_str.strip('[]').split(',')
        return float(parts[0]), float(parts[1])
    except:
        return None, None

X_list, y_list, vids_list = [], [], []
for f in feat_files:
    vid = f.replace('_features.npy', '')
    label_path = LABEL_DIR + '/' + vid + '.csv'
    if not os.path.exists(label_path):
        continue
    
    feats = np.load(FEAT_DIR + '/' + f)
    labels_df = pd.read_csv(label_path)
    n_chunks = len(feats)
    chunk_dur = 5.0
    
    word_times, word_labels = [], []
    for _, row in labels_df.iterrows():
        t0, t1 = parse_timestamp(row['timestamp'])
        if t0 is not None:
            word_times.append((t0, t1))
            word_labels.append(str(row['label']).strip())
    
    chunk_labels = []
    for i in range(n_chunks):
        c0, c1 = i * chunk_dur, (i+1) * chunk_dur
        is_laugh = any(
            wl in ['B', 'I', 'L'] and w0 < c1 and w1 > c0
            for (w0, w1), wl in zip(word_times, word_labels)
        )
        chunk_labels.append(1 if is_laugh else 0)
    
    X_list.append(feats)
    y_list.append(np.array(chunk_labels))
    vids_list.extend([vid] * n_chunks)

X = np.vstack(X_list)
y = np.concatenate(y_list)
print('X:', X.shape, 'y:', y.shape)
print('pos rate:', y.mean().round(3))
print('pos count:', int(y.sum()), 'neg count:', int((y==0).sum()))


In [ ]:
# 4. Train XGBoost and check probabilities
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

neg = (y == 0).sum()
pos = (y == 1).sum()
scale = neg / max(pos, 1)
print('scale_pos_weight:', round(scale, 2))

model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=scale,
    use_label_encoder=False,
    eval_metric='logloss',
    verbosity=0
)
model.fit(X_scaled, y)
print('Model trained')

# Check probability distribution
probs = model.predict_proba(X_scaled)[:, 1]
print('Prob stats:')
print('  min:', probs.min().round(4))
print('  max:', probs.max().round(4))
print('  mean:', probs.mean().round(4))
print('  std:', probs.std().round(4))
print('  median:', np.median(probs).round(4))

# Count by threshold
for th in [0.3, 0.4, 0.5, 0.6, 0.7]:
    n_pos = (probs >= th).sum()
    print(f'  >= {th}: {n_pos} ({100*n_pos/len(probs):.1f}%)')


In [ ]:
# 5. Per-video check: how many predicted segments?
idx = 0
results = []
for f in feat_files:
    vid = f.replace('_features.npy', '')
    label_path = LABEL_DIR + '/' + vid + '.csv'
    if not os.path.exists(label_path):
        continue
    
    feats = np.load(FEAT_DIR + '/' + f)
    n_chunks = len(feats)
    probs = model.predict_proba(scaler.transform(feats))[:, 1]
    
    # Count above threshold
    n_above = (probs >= 0.5).sum()
    
    # Count gt laugh chunks
    labels_df = pd.read_csv(label_path)
    gt_laugh = 0
    for _, row in labels_df.iterrows():
        if str(row['label']).strip() in ['B', 'I', 'L']:
            gt_laugh += 1
    
    results.append({
        'vid': vid,
        'n_chunks': n_chunks,
        'n_pred_above': int(n_above),
        'gt_laugh_words': int(gt_laugh)
    })
    idx += n_chunks

results_df = pd.DataFrame(results)
print('Total videos:', len(results_df))
print('Total chunks with pred >= 0.5:', results_df['n_pred_above'].sum())
print('Total gt laugh words:', results_df['gt_laugh_words'].sum())
print('')
print('Per-video sample:')
print(results_df.head(10).to_string())
